# Экспорт "Сводный отчет" (summary report) из Excel (Excel) в CSV (CSV)

Этот ноутбук (notebook) читает `.xlsx` (xlsx), проходит по блокам подразделений (division blocks) на первом листе (sheet)
и выгружает месяцы (months) + две строки нагрузки (workload):

- Нагрузка (ML), % (ML workload, %)
- Нагрузка, % (manual workload, %)

Результат сохраняется в `../data/processed/excel_svod_workload.csv`.


In [ ]:
from pathlib import Path
import pandas as pd

from excel_svod_parser import parse_svod_first_sheet_from_xlsx_bytes

XLSX_PATH = Path("../data/raw/svod.xlsx")  # положи сюда файл из контура
OUT_PATH = Path("../data/processed/excel_svod_workload.csv")

XLSX_PATH, OUT_PATH


In [ ]:
xlsx_bytes = XLSX_PATH.read_bytes()

parsed = parse_svod_first_sheet_from_xlsx_bytes(
    xlsx_bytes,
    month_header_row=3,
    month_start_col=3,   # C
    blocks_start_row=18,
    block_height=13,
    block_gap=2,
    title_col=2,         # B
    ml_row_in_block=11,
    manual_row_in_block=12,
)

len(parsed), list(parsed.keys())[:5]


In [ ]:
rows = []
for div, rec in parsed.items():
    for month, ml, manual in zip(rec.months, rec.workload_ml_pct, rec.workload_manual_pct):
        rows.append({
            "division_title": div,
            "month": month,
            "workload_ml_pct": ml,
            "workload_manual_pct": manual,
        })

df_excel = pd.DataFrame(rows)
df_excel["month"] = pd.to_datetime(df_excel["month"])

df_excel.head(), df_excel["division_title"].nunique()


In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_excel.to_csv(OUT_PATH, index=False)
OUT_PATH
